# 面试问题：怎样设计可靠的 Agent Tool Calling？

可直接复述的回答：工具调用先是受约束的事务协议，其次才是模型生成 JSON。Schema 要拒绝未知字段、错误类型和非法枚举；授权必须使用服务端主体与资源归属，不能相信模型自报身份。产生副作用的工具需要审批、幂等键和可重放账本。重试只能重放同一意图，不能重复扣款或退款。执行后还要权威回读状态，并把计划、调用、结果和错误写入 trace。任何验证失败都应 fail closed。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：退款工具请求与输入预览

六条脱敏退款请求保留调用 ID、服务端主体、订单、金额、币种、审批票据和幂等键。案例包含正常调用、重复投递、越权订单、非法币种和大额未审批。


In [1]:
calls04 = [  # 构造具有事务语义的退款请求。
    {"call_id": "c1", "subject": "u-17", "order_id": "o-100", "amount": 39.0, "currency": "CNY", "approval": None, "idempotency_key": "refund-o100-v1"},  # 正常小额退款。
    {"call_id": "c2", "subject": "u-17", "order_id": "o-100", "amount": 39.0, "currency": "CNY", "approval": None, "idempotency_key": "refund-o100-v1"},  # 同一意图的重复投递。
    {"call_id": "c3", "subject": "u-22", "order_id": "o-101", "amount": 25.0, "currency": "CNY", "approval": None, "idempotency_key": "refund-o101-v1"},  # 主体与订单归属不一致。
    {"call_id": "c4", "subject": "u-17", "order_id": "o-102", "amount": 1200.0, "currency": "CNY", "approval": None, "idempotency_key": "refund-o102-v1"},  # 大额退款缺少审批。
    {"call_id": "c5", "subject": "u-17", "order_id": "o-102", "amount": 1200.0, "currency": "CNY", "approval": "ticket-fin-9", "idempotency_key": "refund-o102-v2"},  # 大额退款携带审批票据。
    {"call_id": "c6", "subject": "u-17", "order_id": "o-103", "amount": 18.0, "currency": "BTC", "approval": None, "idempotency_key": "refund-o103-v1"},  # 非法币种用于 schema 失败案例。
]  # 完成六条工具调用事件。
owners04 = {"o-100": "u-17", "o-101": "u-31", "o-102": "u-17", "o-103": "u-17"}  # 模拟服务端权威订单归属。
print("教学实验输入：退款工具调用")  # 标识输入预览。
for call04 in calls04:  # 逐条展示动作参数与事务字段。
    print(call04)  # 输出一条退款调用。


教学实验输入：退款工具调用
{'call_id': 'c1', 'subject': 'u-17', 'order_id': 'o-100', 'amount': 39.0, 'currency': 'CNY', 'approval': None, 'idempotency_key': 'refund-o100-v1'}
{'call_id': 'c2', 'subject': 'u-17', 'order_id': 'o-100', 'amount': 39.0, 'currency': 'CNY', 'approval': None, 'idempotency_key': 'refund-o100-v1'}
{'call_id': 'c3', 'subject': 'u-22', 'order_id': 'o-101', 'amount': 25.0, 'currency': 'CNY', 'approval': None, 'idempotency_key': 'refund-o101-v1'}
{'call_id': 'c4', 'subject': 'u-17', 'order_id': 'o-102', 'amount': 1200.0, 'currency': 'CNY', 'approval': None, 'idempotency_key': 'refund-o102-v1'}
{'call_id': 'c5', 'subject': 'u-17', 'order_id': 'o-102', 'amount': 1200.0, 'currency': 'CNY', 'approval': 'ticket-fin-9', 'idempotency_key': 'refund-o102-v2'}
{'call_id': 'c6', 'subject': 'u-17', 'order_id': 'o-103', 'amount': 18.0, 'currency': 'BTC', 'approval': None, 'idempotency_key': 'refund-o103-v1'}


## 2. Baseline（基线）：只要字段能取到就执行

朴素执行器既不校验 schema，也没有幂等账本，重复投递会产生两次副作用。这里用累计退款金额展示错误，而不真正调用支付系统。


In [2]:
baseline_refunds04 = {}  # 保存朴素执行器产生的退款总额。
for call04 in calls04[:2]:  # 连续执行原调用和重复投递。
    order04 = call04["order_id"]  # 读取模型提供的订单号。
    baseline_refunds04[order04] = baseline_refunds04.get(order04, 0.0) + call04["amount"]  # 未做幂等检查直接累加副作用。
print("基线账本", baseline_refunds04)  # 展示重复退款后的错误金额。
print("基线结果：订单 o-100 被退款两次，累计", baseline_refunds04["o-100"])  # 明确指出重复副作用。


基线账本 {'o-100': 78.0}
基线结果：订单 o-100 被退款两次，累计 78.0


## 3. 核心实现：严格 schema、授权、审批与幂等账本

验证顺序是字段集合、类型与范围、服务端资源归属、审批票据，最后才查询幂等账本。已成功的幂等键返回原结果；失败请求不污染账本。


In [3]:
required_fields04 = {"call_id", "subject", "order_id", "amount", "currency", "approval", "idempotency_key"}  # 冻结允许的工具字段集合。
ledger04 = {}  # 保存成功调用的幂等结果。
authoritative_refunds04 = {}  # 模拟支付系统的权威退款状态。
events04 = []  # 收集每次调用的审计事件。
def execute_refund04(call04):  # 实现严格的退款事务入口。
    if set(call04) != required_fields04:  # 拒绝缺失或额外字段。
        return {"status": "rejected", "reason": "schema_fields"}  # 返回稳定 schema 错误码。
    if not isinstance(call04["amount"], float) or call04["amount"] <= 0.0:  # 检查金额类型与范围。
        return {"status": "rejected", "reason": "amount_contract"}  # 拒绝非法金额。
    if call04["currency"] != "CNY":  # 限定教学工具只处理人民币。
        return {"status": "rejected", "reason": "currency_enum"}  # 拒绝非法枚举。
    if owners04.get(call04["order_id"]) != call04["subject"]:  # 使用权威归属而非模型声明授权。
        return {"status": "rejected", "reason": "resource_forbidden"}  # 拒绝跨主体订单。
    if call04["amount"] > 500.0 and not call04["approval"]:  # 检查大额退款审批门槛。
        return {"status": "rejected", "reason": "approval_required"}  # 阻止未审批的大额动作。
    key04 = call04["idempotency_key"]  # 读取稳定业务幂等键。
    if key04 in ledger04:  # 检查同一意图是否已经完成。
        return {"status": "replayed", "result": ledger04[key04]}  # 返回原结果而不重复执行。
    authoritative_refunds04[call04["order_id"]] = call04["amount"]  # 模拟一次真实退款提交。
    result04 = {"order_id": call04["order_id"], "refunded": call04["amount"]}  # 构造权威结果摘要。
    ledger04[key04] = result04  # 原子记录幂等键与结果。
    return {"status": "committed", "result": result04}  # 返回成功事务结果。
for call04 in calls04:  # 依次执行正常与异常工具请求。
    outcome04 = execute_refund04(call04)  # 通过严格事务入口处理请求。
    events04.append((call04["call_id"], outcome04["status"], outcome04.get("reason", "ok")))  # 保存调用状态与原因码。
print("工具事件账本：call_id | status | reason")  # 输出核心状态轨迹表头。
for event04 in events04:  # 逐条展示事务决策。
    print(event04)  # 输出提交、重放或拒绝事件。


工具事件账本：call_id | status | reason
('c1', 'committed', 'ok')
('c2', 'replayed', 'ok')
('c3', 'rejected', 'resource_forbidden')
('c4', 'rejected', 'approval_required')
('c5', 'committed', 'ok')
('c6', 'rejected', 'currency_enum')


## 4. 结果表与结果解读

严格执行器只提交 `c1` 和携带审批的 `c5`；`c2` 返回原结果，其他请求分别因越权、缺审批和非法币种被拒绝。幂等不是简单去重请求 ID，而是把业务意图键绑定到已提交结果。


In [4]:
status_counts04 = {}  # 汇总各事务状态数量。
for _, status04, _ in events04:  # 遍历审计事件。
    status_counts04[status04] = status_counts04.get(status04, 0) + 1  # 累计当前状态出现次数。
print("执行器 | o-100累计退款 | 已提交订单 | 状态分布")  # 输出基线和核心方案对照表头。
print("naive", baseline_refunds04["o-100"], list(baseline_refunds04), {"committed": 2})  # 展示重复副作用。
print("strict", authoritative_refunds04["o-100"], sorted(authoritative_refunds04), status_counts04)  # 展示严格事务结果。
print("结果解读：重复投递被重放，失败调用没有改变权威退款状态")  # 解释幂等和 fail-closed 效果。


执行器 | o-100累计退款 | 已提交订单 | 状态分布
naive 78.0 ['o-100'] {'committed': 2}
strict 39.0 ['o-100', 'o-102'] {'committed': 2, 'replayed': 1, 'rejected': 3}
结果解读：重复投递被重放，失败调用没有改变权威退款状态


## 5. 失败案例与修正：重试导致重复退款

网络超时后客户端通常会重试。失败方案用新 `call_id` 当幂等依据，于是同一订单退款两次；修正方案使用稳定业务键 `refund-o100-v1`，并返回第一次提交的原结果。


In [5]:
duplicate_event04 = events04[1]  # 读取重复投递的严格执行结果。
original_result04 = ledger04["refund-o100-v1"]  # 读取第一次提交的权威结果。
print("失败行为：按 call_id 去重会把 c1 与 c2 当成两次退款")  # 展示错误幂等粒度。
print("修正行为", duplicate_event04, "原结果", original_result04)  # 展示业务幂等键的重放语义。
print("权威回读", authoritative_refunds04)  # 输出执行后的最终外部状态。


失败行为：按 call_id 去重会把 c1 与 c2 当成两次退款
修正行为 ('c2', 'replayed', 'ok') 原结果 {'order_id': 'o-100', 'refunded': 39.0}
权威回读 {'o-100': 39.0, 'o-102': 1200.0}


## 6. 生产边界与工具合同

真实系统要把幂等写入事务数据库，并绑定参数摘要，防止同一键对应不同金额。审批票据要校验签名、主体、动作和有效期；工具凭据必须短期且最小权限。


In [6]:
tool_contract04 = {"tool": "refund_order", "schema": 3, "idempotency_scope": "subject+order+intent", "approval_threshold": 500.0, "audit_retention_days": 180}  # 定义生产工具协议要素。
print("工具发布合同", tool_contract04)  # 展示 schema 与事务策略版本。
print("生产替换点：事务数据库、审批验签、参数摘要、权威状态回读和告警")  # 说明教学内存字典的局限。


工具发布合同 {'tool': 'refund_order', 'schema': 3, 'idempotency_scope': 'subject+order+intent', 'approval_threshold': 500.0, 'audit_retention_days': 180}
生产替换点：事务数据库、审批验签、参数摘要、权威状态回读和告警


## 7. 最小回归测试

断言只保护副作用、重放和关键拒绝原因。


In [7]:
assert len(calls04) >= 5  # 保证案例覆盖正常与异常调用。
assert authoritative_refunds04["o-100"] == 39.0  # 保证重复投递没有造成双重退款。
assert duplicate_event04[1] == "replayed"  # 保证同一业务意图返回原结果。
assert dict((event04[0], event04[2]) for event04 in events04)["c3"] == "resource_forbidden"  # 保证越权订单被拒绝。
assert dict((event04[0], event04[2]) for event04 in events04)["c6"] == "currency_enum"  # 保证非法币种被 schema 门禁拒绝。
print("最小回归测试通过：schema、授权、审批和幂等关键路径稳定")  # 显示事务安全检查已经完成。


最小回归测试通过：schema、授权、审批和幂等关键路径稳定
